# 🚀 01. Train Multi-Modal Teacher Model (Phase 1B - CMKD)

> **Project:** Durian Ripeness Classification via Cross-Modal Knowledge Distillation (CMKD)  
> **Lead:** Leader / Member 1 (Vision Lead & Multi-Modal Framework)  
> **Notebook Role:** Lightweight launcher notebook used to:
1. Mount Google Drive & verify GPU availability.
2. `git clone` / `git pull` the latest source code from GitHub.
3. Link dataset (`data/final`) and unimodal checkpoints (`checkpoints/unimodal/{vision,audio,sensor}`) from Google Drive, and prepare target `checkpoints/teachermodal`.
4. Launch `train_teacher.py` from the repository root.

In [13]:
# ==============================================================================
# 1. MOUNT GOOGLE DRIVE & CHECK GPU
# ==============================================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    print("✅ Google Drive mounted successfully!")
else:
    print("ℹ️ Running in Local environment.")

!nvidia-smi

✅ Google Drive mounted successfully!
Fri Nov 20 18:41:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------

In [14]:
# ==============================================================================
# 2. SYNC LATEST CODE FROM GITHUB
# ==============================================================================
REPO_URL = "https://github.com/Cocobaut/durian-ripeness-classification.git"
# Multi-View ConvNeXtV2 Teacher code lives on branch `cocobaut` (not merged into `main` yet).
# `main` still expects the old `best_vision_unimodal.pth` -> FileNotFoundError.
BRANCH = "cocobaut"
REPO_DIR = "/content/durian-ripeness-classification"

if IN_COLAB:
    %cd /content
    if not os.path.exists(REPO_DIR):
        if BRANCH:
            !git clone -b $BRANCH $REPO_URL $REPO_DIR
        else:
            !git clone $REPO_URL $REPO_DIR
        %cd $REPO_DIR
    else:
        %cd $REPO_DIR
        # Remove temporary repo symlinks (without touching Drive files) to prevent git pull conflicts
        !find . -maxdepth 5 -type l -delete
        !git fetch origin
        if BRANCH:
            !git checkout -f -B $BRANCH origin/$BRANCH
            !git reset --hard origin/$BRANCH
        else:
            !git reset --hard origin/$(git rev-parse --abbrev-ref HEAD)
    !git log --oneline -1
else:
    # If running locally from a subfolder, walk up to the repository root
    cur = os.path.abspath(".")
    while not (os.path.isdir(os.path.join(cur, "src")) and os.path.isdir(os.path.join(cur, "workspaces"))):
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    os.chdir(cur)

print(f"✅ Working in project root directory: {os.getcwd()}")

/content
/content/durian-ripeness-classification
branch 'cocobaut' set up to track 'origin/cocobaut'.
Switched to a new branch 'cocobaut'
HEAD is now at ddb1ea7 [UPDATE] teacher and student
ddb1ea7 (HEAD -> cocobaut, origin/cocobaut) [UPDATE] teacher and student
✅ Working in project root directory: /content/durian-ripeness-classification


In [15]:
# ==============================================================================
# 3. FAST DEPENDENCY INSTALLATION WITH `uv`
# ==============================================================================
!pip install -q uv
!uv pip install --system -r requirements.txt
print("✅ Installed all dependencies successfully!")

Using Python 3.13.16 environment at: /usr
Checked 22 packages in 102ms
✅ Installed all dependencies successfully!


In [16]:
# ==============================================================================
# 4. CONFIGURE CHECKPOINT & DATASET LINKS FROM GOOGLE DRIVE
# ==============================================================================
# Drive Checkpoints folder: https://drive.google.com/drive/folders/1K6II5aFG31kYrl7rsJj53u2Np4ZzjzG_?usp=drive_link
# Contains: unimodal/{vision, audio, sensor} and teachermodal/
from pathlib import Path

if IN_COLAB:
    # Adjust this path to match your Google Drive location
    DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/durian-ripeness-cmkd")
    DRIVE_CHECKPOINTS = DRIVE_PROJECT_ROOT / "checkpoints"
    DRIVE_DATA = DRIVE_PROJECT_ROOT / "data" / "final"
    DRIVE_RESULTS = DRIVE_PROJECT_ROOT / "results_teacher"

    # Pre-create teachermodal & results_teacher folders on Google Drive for real-time persistence
    (DRIVE_CHECKPOINTS / "teachermodal").mkdir(parents=True, exist_ok=True)
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

    # Symlink repo directories to Google Drive so outputs persist across Colab disconnects
    !rm -rf checkpoints/unimodal checkpoints/teachermodal workspaces/member1_vision_lead/04_train_teacher_and_student/results
    !mkdir -p checkpoints data/final
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'unimodal')}" checkpoints/unimodal
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'teachermodal')}" checkpoints/teachermodal
    !ln -sfn "{str(DRIVE_RESULTS)}" workspaces/member1_vision_lead/04_train_teacher_and_student/results
    for split_name in ["train", "valid", "test"]:
        src_split = DRIVE_DATA / split_name
        if src_split.exists():
            !rm -rf data/final/{split_name}
            !ln -sfn "{str(src_split)}" data/final/{split_name}

Path("checkpoints/teachermodal").mkdir(parents=True, exist_ok=True)

# Unimodal checkpoints used by the Teacher (vision = new Multi-View ConvNeXtV2 256-d extractor)
VISION_CKPT = "checkpoints/unimodal/vision/convnextv2_multiview_backbone_256d.pth"
AUDIO_CKPT = "checkpoints/unimodal/audio/audio_backbone_teacher.pth"
SENSOR_CKPT = "checkpoints/unimodal/sensor/best_sensor_unimodal.pth"

print("✅ Verifying configured paths:")
print(" - Unimodal dir   :", Path("checkpoints/unimodal").resolve())
print(" - Teachermodal   :", Path("checkpoints/teachermodal").resolve())
print(" - Manifest CSV   :", Path("data/final/final_multimodal_manifest.csv").exists())
for name, p in [("Vision ckpt", VISION_CKPT), ("Audio ckpt", AUDIO_CKPT), ("Sensor ckpt", SENSOR_CKPT)]:
    ok = Path(p).is_file()
    print(f" - {name:<14}:", "✅" if ok else "❌ MISSING", p)
    assert ok, f"{name} not found: {p}"
print(" - Vision folder  :", sorted(x.name for x in Path("checkpoints/unimodal/vision").iterdir()))
print(" - Teachermodal   :", sorted(x.name for x in Path("checkpoints/teachermodal").iterdir()))

✅ Verifying configured paths:
 - Unimodal dir   : /content/drive/MyDrive/durian-ripeness-cmkd/checkpoints/unimodal
 - Teachermodal   : /content/drive/MyDrive/durian-ripeness-cmkd/checkpoints/teachermodal
 - Manifest CSV   : True
 - Vision ckpt   : ✅ checkpoints/unimodal/vision/convnextv2_multiview_backbone_256d.pth
 - Audio ckpt    : ✅ checkpoints/unimodal/audio/audio_backbone_teacher.pth
 - Sensor ckpt   : ✅ checkpoints/unimodal/sensor/best_sensor_unimodal.pth
 - Vision folder  : ['best_convnextv2_multiview.pth', 'convnextv2_multiview_backbone_256d.pth', 'convnextv2_pretrained_raw_weights.pth', 'last_convnextv2_multiview.pth']
 - Teachermodal   : []


In [17]:
# ==============================================================================
# 5. RUN MULTI-MODAL TEACHER TRAINING (MULTI-VIEW CONVNEXTV2 GATED ATTENTION + CROSS-ATTENTION FUSION)
# ==============================================================================
# --fresh: ignore the old checkpoints/teachermodal/last_teacher.pth from the previous run.
#          Without it, the script may resume from that file (already at epoch 30/30) -> skips training
#          and just re-evaluates the OLD teacher. Remove --fresh only to resume THIS run after a disconnect.
!python workspaces/member1_vision_lead/04_train_teacher_and_student/train_teacher.py \
    --manifest data/final/final_multimodal_manifest.csv \
    --data-dir data/final \
    --vision-ckpt {VISION_CKPT} \
    --audio-ckpt {AUDIO_CKPT} \
    --sensor-ckpt {SENSOR_CKPT} \
    --save-path checkpoints/teachermodal/best_teacher.pth \
    --fusion-type cross_attention \
    --loss-type cost_sensitive_focal \
    --focal-gamma 2.0 \
    --cost-weight 0.5 \
    --batch-size 4 \
    --accum-steps 2 \
    --lr-backbone 1.5e-5 \
    --lr-head 5e-4 \
    --epochs 30 \
    --warmup-epochs 5 \
    --select-metric fruit_macro_f1 \
    --eval-test \
    --fresh \
    --device cuda

[2026-11-20 18:41:28] ==============================================================================
[2026-11-20 18:41:28] TRAINING MULTI-MODAL TEACHER (STAGE 1B)
[2026-11-20 18:41:28] Device: cuda | AMP: True (torch.bfloat16) | batch 4 x accum 2
[2026-11-20 18:41:28] Vision ckpt : /content/durian-ripeness-classification/checkpoints/unimodal/vision/convnextv2_multiview_backbone_256d.pth
[2026-11-20 18:41:28] Audio ckpt  : /content/durian-ripeness-classification/checkpoints/unimodal/audio/audio_backbone_teacher.pth
[2026-11-20 18:41:28] Sensor ckpt : /content/durian-ripeness-classification/checkpoints/unimodal/sensor/best_sensor_unimodal.pth
[2026-11-20 18:41:28] Best ckpt   : /content/durian-ripeness-classification/checkpoints/teachermodal/best_teacher.pth
[2026-11-20 18:41:28] Resume ckpt : /content/durian-ripeness-classification/checkpoints/teachermodal/last_teacher.pth (fresh run from Epoch 1)
[2026-11-20 18:41:28] ====================================================================

In [18]:
# ==============================================================================
# 6. INDEPENDENT EVALUATION & BENCHMARK REPORT WITH EVAL_TEACHER
# ==============================================================================
!python -m src.evaluators.eval_teacher \
    --checkpoint checkpoints/teachermodal/best_teacher.pth \
    --manifest data/final/final_multimodal_manifest.csv \
    --test-dir data/final/test \
    --input-mode multimodal \
    --output-dir workspaces/member1_vision_lead/04_train_teacher_and_student/results

print("✅ All evaluation results saved at:", Path("workspaces/member1_vision_lead/04_train_teacher_and_student/results").resolve())

/content/durian-ripeness-classification/src/datasets/sensor_dataset.py:132: UserWarning: Failed to load /content/durian-ripeness-classification/workspaces/member3_sensor/02_train_unimodal_sensor/results/sensor_preprocessor_unimodal.joblib (TypeError); refitting on Train.
  warnings.warn(f"Failed to load {joblib_path} ({type(exc).__name__}); refitting on Train.")
[data] test: extracting knocks from 38 audio files ...
[data] {'split': 'test', 'n_fruits': 29, 'n_samples': 47, 'n_rgb_views': 298, 'pad_views': 18, 'fruits_without_rgb': 5, 'fruits_without_audio': 9, 'fruits_without_sensor': 0}
[2026-11-20 19:21:57] [INFO] [eval_teacher]: Initialized TeacherEvaluator on device: cuda (input_mode=multimodal)
[2026-11-20 19:21:57] [INFO] [eval_teacher]: Evaluation results will be saved to: /content/durian-ripeness-classification/workspaces/member1_vision_lead/04_train_teacher_and_student/results
[2026-11-20 19:21:57] [INFO] [eval_teacher]: === STARTING FULL TEACHER EVALUATION PIPELINE ===
[2026-